In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/rust_model.ot
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/spm.model
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/config.json
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/README.md
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/tf_model.h5
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/tokenizer_config.json
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/gitattributes
/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base/pytorch_model.bin
/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/

In [2]:
# Cell 1: setup
!pip install -q sentencepiece protobuf

from pathlib import Path

WEIGHTS_DIR = Path("/kaggle/input/datasets/sqizeeeeeeeee/final-weights/res")
TEST_PATH = Path("/kaggle/input/competitions/llm-classification-finetuning/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv")
OUTPUT_DIR = Path("/kaggle/working")
BACKBONE_PATH = Path("/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base")


N_FOLDS = 5

assert TEST_PATH.exists(), f"test.csv not found at {TEST_PATH}"
assert all((WEIGHTS_DIR / f"fold{i}" / "best.pt").exists() for i in range(N_FOLDS)), "missing a fold checkpoint"
assert BACKBONE_PATH.exists(), f"backbone not found at {BACKBONE_PATH}"

print(f"[setup] weights: {WEIGHTS_DIR}")
print(f"[setup] test: {TEST_PATH}")

[setup] weights: /kaggle/input/datasets/sqizeeeeeeeee/final-weights/res
[setup] test: /kaggle/input/competitions/llm-classification-finetuning/test.csv


In [3]:
# Cell 2: imports
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModel, AutoTokenizer

def set_seed(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[setup] device: {device}")

[setup] device: cuda


In [4]:
# Cell 3: RewardModel (inference-only) + simple test dataset
class RewardModel(nn.Module):
    def __init__(self, backbone="microsoft/deberta-v3-base", pooling="cls", dropout=0.1, delta_init=0.0):
        super().__init__()
        self.pooling = pooling
        config = AutoConfig.from_pretrained(backbone)
        self.encoder = AutoModel.from_pretrained(backbone, config=config, torch_dtype=torch.float32)
        self.dropout = nn.Dropout(dropout)
        self.reward_head = nn.Linear(config.hidden_size, 1)
        self.delta = nn.Parameter(torch.tensor(float(delta_init)))
        self._eps = 1e-6

    def _pool(self, last_hidden_state, attention_mask):
        if self.pooling == "cls":
            return last_hidden_state[:, 0, :]
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        summed = (last_hidden_state * mask).sum(dim=1)
        counted = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counted

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._pool(out.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        return self.reward_head(pooled).squeeze(-1)

    def forward(self, input_ids, attention_mask):
        rewards = self.encode(input_ids, attention_mask)  # (batch * 2,), interleaved [a0, b0, a1, b1, ...]
        r_a = rewards[0::2]
        r_b = rewards[1::2]
        d = r_a - r_b
        p_a = torch.sigmoid(d - self.delta)
        p_b = torch.sigmoid(-d - self.delta)
        p_tie = (1.0 - p_a - p_b).clamp(min=self._eps)
        return {"r_a": r_a, "r_b": r_b, "d": d, "p_a": p_a, "p_b": p_b, "p_tie": p_tie}


class TestPairDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row["prompt"]
        enc_a = self.tokenizer(text=prompt, text_pair=row["response_a"], truncation="longest_first", max_length=self.max_length)
        enc_b = self.tokenizer(text=prompt, text_pair=row["response_b"], truncation="longest_first", max_length=self.max_length)
        return {
            "input_ids_a": enc_a["input_ids"], "attention_mask_a": enc_a["attention_mask"],
            "input_ids_b": enc_b["input_ids"], "attention_mask_b": enc_b["attention_mask"],
        }


class TestCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids, attention_mask = [], []
        for item in batch:
            input_ids.append(item["input_ids_a"])
            input_ids.append(item["input_ids_b"])
            attention_mask.append(item["attention_mask_a"])
            attention_mask.append(item["attention_mask_b"])
        padded = self.tokenizer.pad({"input_ids": input_ids, "attention_mask": attention_mask}, padding=True, return_tensors="pt")
        return {"input_ids": padded["input_ids"], "attention_mask": padded["attention_mask"]}


print("[setup] RewardModel + TestPairDataset + TestCollator defined")

[setup] RewardModel + TestPairDataset + TestCollator defined


In [5]:
# Cell 4: load and parse test.csv
EMPTY_TURN = "[EMPTY]"

def parse_turns(x: str) -> list[str]:
    parsed = json.loads(x)
    if not isinstance(parsed, list):
        raise ValueError(f"expected a JSON list, got {type(parsed).__name__}: {x[:100]!r}")
    return [t if isinstance(t, str) and t.strip() else EMPTY_TURN for t in parsed]


def sanitize_text(text: str) -> str:
    return text.encode("utf-8", errors="surrogatepass").decode("utf-8", errors="replace")


test_df = pd.read_csv(TEST_PATH)
print(f"[data] {len(test_df)} rows from {TEST_PATH}")

for col in ["prompt", "response_a", "response_b"]:
    parsed = test_df[col].apply(parse_turns)
    test_df[col] = parsed.apply(lambda turns: sanitize_text(" ".join(turns).strip()))

print(test_df.columns.tolist())
test_df.head(2)

[data] 3 rows from /kaggle/input/competitions/llm-classification-finetuning/test.csv
['id', 'prompt', 'response_a', 'response_b']


,id,prompt,response_a,response_b
0,136060,"I have three oranges today, I ate an orange ye...",You have two oranges today.,You still have three oranges. Eating an orange...
1,211333,You are a mediator in a heated political debat...,Thank you for sharing the details of the situa...,Mr Reddy and Ms Blue both have valid points in...


In [6]:
# Cell 5: run all 5 folds, average predictions, build submission.csv
tokenizer = AutoTokenizer.from_pretrained(str(BACKBONE_PATH))
MAX_LENGTH = 1024

test_ds = TestPairDataset(test_df, tokenizer, MAX_LENGTH)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=TestCollator(tokenizer))

all_fold_probs = []  # list of (n_rows, 3) arrays, one per fold

for fold in range(N_FOLDS):
    print(f"[infer] fold {fold}")
    model = RewardModel(backbone=str(BACKBONE_PATH)).to(device)
    model.load_state_dict(torch.load(WEIGHTS_DIR / f"fold{fold}" / "best.pt", map_location=device))
    model.eval()

    fold_probs = []
    with torch.inference_mode():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            out = model(input_ids, attention_mask)
            probs = torch.stack([out["p_a"], out["p_b"], out["p_tie"]], dim=1)
            fold_probs.append(probs.cpu().numpy())

    all_fold_probs.append(np.concatenate(fold_probs, axis=0))
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

avg_probs = np.mean(all_fold_probs, axis=0)  # (n_rows, 3), averaged across 5 folds
print(f"[infer] averaged predictions shape: {avg_probs.shape}")

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
print("[check] sample_submission columns:", sample_sub.columns.tolist())

[transformers] The tokenizer you are loading from '/kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[infer] fold 0


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: /kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if 

[infer] fold 1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

... (7 outputs left out)

[infer] fold 4


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: /kaggle/input/models/gauripatil2299/deberta-v3-base/transformers/default/1/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if 

[infer] averaged predictions shape: (3, 3)
[check] sample_submission columns: ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']


In [7]:
# Cell 6: build and save submission.csv
submission = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": avg_probs[:, 0],
    "winner_model_b": avg_probs[:, 1],
    "winner_tie": avg_probs[:, 2],
})

print(submission)

# sanity check: probabilities should sum to ~1 per row
row_sums = submission[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print(f"[check] row sums (should be ~1.0): {row_sums.tolist()}")

submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print(f"[save] wrote submission to {OUTPUT_DIR / 'submission.csv'}")

        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.450138        0.449341    0.100520
1   211333        0.449938        0.449542    0.100520
2  1233961        0.447181        0.452312    0.100507
[check] row sums (should be ~1.0): [1.0, 1.0, 1.0000001192092896]
[save] wrote submission to /kaggle/working/submission.csv
